# Privacy Risk Factors

Analysis for permissions, trackers, and their relationships with privacy metrics.

# Common setup and data preparation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

metrics_results = Path("../data/mhealth_apps_metrics.csv")
if not metrics_results.exists():
    raise FileNotFoundError(
        "Could not find `mhealth_apps_metrics.csv` in the notebook folder or at ../data/."
    )

REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", 
    "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", 
    "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", 
    "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", 
    "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", 
    "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", 
    "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina",
    "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain",
    "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia",
    "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore",
    "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria",
    "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand",
    "pg": "Papua New Guinea"
}

REGION_ORDER = [
    "North America", "Latin America", "Europe",
    "Middle East", "Asia", "Africa", "Oceania"
]

region_palette = plt.cm.tab10(np.linspace(0, 1, len(REGION_ORDER)))
REGION_COLORS = {region: region_palette[i] for i, region in enumerate(REGION_ORDER)}
category_palette = plt.cm.tab20(np.linspace(0, 1, 20))


In [ ]:
import pandas as pd

df = pd.read_csv(metrics_results, low_memory=False)
print("Raw shape:", df.shape)

required_metrics = ["ADII", "DGI", "PCLR", "AS"]
missing_required = [c for c in required_metrics if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required metric columns: {missing_required}")

analysis_df = df.copy()

print("Analysis shape:", analysis_df.shape)
print("Unique apps:", analysis_df["app_id"].nunique())

display_cols = [
    "app_id", "country", "country_label", "region", "category",
    "ADII", "DGI", "PCLR", "AS",
    "observed_count", "disclosed_count", "missing_count", "misleading_count"
]
display_cols = [c for c in display_cols if c in analysis_df.columns]

In [ ]:

summary = pd.DataFrame({
    "non_null": analysis_df[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": analysis_df[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": analysis_df[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": analysis_df[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": analysis_df[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": analysis_df[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)
summary


In [ ]:

country_level = analysis_df.copy()

metric_agg = {
    "ADII": "mean",
    "DGI": "mean",
    "PCLR": "mean",
    "AS": "mean",
}

meta_agg = {
    "country_label": "nunique",
    "region": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "category": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "num_permissions": "mean",
    "num_dangerous_permissions": "mean",
    "num_trackers": "mean",
    "downloads_int": "mean",
    "average_score": "mean",
}

agg_dict = {}
for k, v in {**metric_agg, **meta_agg}.items():
    if k in analysis_df.columns:
        agg_dict[k] = v

app_level = analysis_df.groupby("app_id", as_index=False).agg(agg_dict)
app_level = app_level.rename(columns={"country_label": "country_count"})
app_level_all = app_level.copy()

if "PCLR" not in app_level_all.columns:
    if "app_country_PCLR" in analysis_df.columns:
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)["app_country_PCLR"]
            .mean()
            .rename(columns={"app_country_PCLR": "PCLR"})
        )
        app_level_all = app_level_all.merge(pclr_fallback, on="app_id", how="left")
    elif {"app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"}.issubset(analysis_df.columns):
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)[
                ["app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"]
            ]
            .mean()
        )
        denom = pclr_fallback["app_country_total_sensitive_instances"].replace(0, np.nan)
        pclr_fallback["PCLR"] = pclr_fallback["app_country_pre_sensitive_instances"] / denom
        app_level_all = app_level_all.merge(pclr_fallback[["app_id", "PCLR"]], on="app_id", how="left")

print("Country-level rows:", len(country_level))
print("App-level rows:", len(app_level_all))
print("App-level metric columns:", [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in app_level_all.columns])
app_level = app_level_all.copy()




## 1. Figure — Scatter view of invasiveness vs disclosure gap

This view helps reveal whether more invasive apps also tend to have larger disclosure gaps.
Category is encoded by color, and point size encodes the number of countries covered.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plot_df = app_level_all.copy() if "app_level_all" in globals() else app_level.copy()

metric_aliases = {
    "ADII": ["ADII", "ADII_mean", "ADII_median"],
    "DGI": ["DGI", "DGI_mean", "DGI_median"],
    "country_count": ["country_count", "n_countries"],
}
for target, candidates in metric_aliases.items():
    if target not in plot_df.columns:
        for cand in candidates:
            if cand in plot_df.columns:
                plot_df[target] = plot_df[cand]
                break

required_cols = ["category", "ADII", "DGI", "country_count"]
missing = [c for c in required_cols if c not in plot_df.columns]
if missing:
    raise KeyError(f"Missing required columns for plotting: {missing}")

plot_df = plot_df.dropna(subset=["category", "ADII", "DGI"]).copy()

top_categories = plot_df["category"].value_counts().head(10).index.tolist()
plot_df["category_simple"] = plot_df["category"].where(
    plot_df["category"].isin(top_categories), "Other"
)

all_plot_categories = plot_df["category_simple"].unique().tolist()
extra_palette = plt.cm.tab20b(np.linspace(0, 1, len(all_plot_categories)))
plot_category_colors = {cat: extra_palette[i] for i, cat in enumerate(all_plot_categories)}

fig, ax = plt.subplots(figsize=(5.5, 3.5))

for cat, sub in plot_df.groupby("category_simple"):
    ax.scatter(
        sub["ADII"],
        sub["DGI"],
        s=10 + 4 * pd.to_numeric(sub["country_count"], errors="coerce").fillna(1).clip(lower=1),
        alpha=0.65,
        label=cat,
        color=plot_category_colors[cat],
        edgecolor="white",
        linewidth=0.35
    )

corr = plot_df[["ADII", "DGI"]].corr().iloc[0, 1]

ax.set_title(
    f"ADII vs DGI Across Apps (corr = {corr:.2f})",
    fontsize=10,
    weight="bold"
)
ax.set_xlabel("Mean ADII", fontsize=9)
ax.set_ylabel("Mean DGI", fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, linestyle="--", alpha=0.3)

ax.legend(
    title="Category",
    fontsize=7,
    title_fontsize=8,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False
    print("statsmodels is not available; regression cells will be skipped.")

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

if "app_level_all" in globals():
    biz_df = app_level_all.copy()
elif "app_level" in globals():
    biz_df = app_level.copy()
elif "analysis_df" in globals():
    biz_df = analysis_df.drop_duplicates(subset=["app_id"]).copy()
else:
    raise NameError("Expected one of app_level_all, app_level, or analysis_df to exist.")

print(f"Initial shape: {biz_df.shape}")

In [ ]:
if "paid_app" not in biz_df.columns:
    if "free" in biz_df.columns:
        biz_df["paid_app"] = (~biz_df["free"].fillna(True)).astype(int)
    elif "price" in biz_df.columns:
        biz_df["paid_app"] = (
            pd.to_numeric(biz_df["price"], errors="coerce")
            .fillna(0)
            .gt(0)
            .astype(int)
        )

if "offersIAP" in biz_df.columns and "has_iap" not in biz_df.columns:
    biz_df["has_iap"] = biz_df["offersIAP"].fillna(False).astype(int)

if "ad_supported" in biz_df.columns and "has_ads" not in biz_df.columns:
    biz_df["has_ads"] = biz_df["ad_supported"].fillna(False).astype(int)

if "downloads_int" in biz_df.columns:
    biz_df["downloads_int"] = pd.to_numeric(biz_df["downloads_int"], errors="coerce")
    biz_df["log_downloads"] = np.log1p(biz_df["downloads_int"])

metric_candidates = {
    "ADII": ["ADII", "ADII_mean"],
    "DGI": ["DGI", "DGI_mean"],
    "PCLR": ["PCLR", "PCLR_mean"],
    "AS": ["AS", "AS_mean"]
}

for target, candidates in metric_candidates.items():
    if target not in biz_df.columns:
        for cand in candidates:
            if cand in biz_df.columns:
                biz_df[target] = biz_df[cand]
                break

numeric_cols = [
    "ADII", "DGI", "PCLR", "AS",
    "num_trackers", "num_permissions", "num_dangerous_permissions",
    "average_score", "log_downloads", "downloads_int"
]
for c in numeric_cols:
    if c in biz_df.columns:
        biz_df[c] = pd.to_numeric(biz_df[c], errors="coerce")

if "top_grossing" in biz_df.columns:
    biz_df["top_grossing_flag"] = (
        biz_df["top_grossing"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["yes", "true", "1"])
        .astype(int)
    )

keep_cols = [c for c in [
    "app_id", "app_name", "category", "categories", "region",
    "has_ads", "has_iap", "paid_app",
    "num_trackers", "num_permissions", "num_dangerous_permissions",
    "downloads_int", "log_downloads", "average_score", "top_grossing_flag",
    "ADII", "DGI", "PCLR", "AS"
] if c in biz_df.columns]

biz_df = biz_df[keep_cols].copy()

print("Prepared shape:", biz_df.shape)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from pathlib import Path

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
})

FIG_DIR = Path(FIG_DIR) if "FIG_DIR" in globals() else Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

fig_df = biz_df.copy()

numeric_cols = [
    "num_dangerous_permissions",
    "PCLR",
    "num_trackers",
    "ADII",
    "num_permissions",
    "DGI",
]
for col in numeric_cols:
    if col in fig_df.columns:
        fig_df[col] = pd.to_numeric(fig_df[col], errors="coerce")


region_color_map = {
    "Europe": "tab:blue",
    "North America": "tab:orange",
    "Asia": "tab:green",
    "Africa": "tab:red",
    "Latin America": "tab:purple",
    "Middle East": "tab:brown",
    "Oceania": "tab:pink",
}

default_color = "tab:gray"
if "region" in fig_df.columns:
    point_colors = fig_df["region"].map(region_color_map).fillna(default_color)
else:
    point_colors = pd.Series([default_color] * len(fig_df), index=fig_df.index)


def add_regression_line(ax, x, y):
    valid = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valid) >= 2 and valid["x"].nunique() > 1:
        z = np.polyfit(valid["x"], valid["y"], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(valid["x"].values)
        ax.plot(x_sorted, p(x_sorted), linewidth=2)

def add_corr_text(ax, x, y, method="pearson"):
    # ADII is strongly right-skewed (skewness ~9.6; max is ~73x the
    # median), so Pearson correlation involving ADII understates/distorts
    # the relationship and is sensitive to a small number of outliers
    # (verified: Pearson r on trackers-vs-ADII drops noticeably once the
    # top 1% of ADII values are excluded). Spearman correlation, which only
    # depends on rank order, is used for any panel involving ADII instead.
    valid = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valid) >= 3 and valid["x"].nunique() > 1:
        if method == "spearman":
            r, p = spearmanr(valid["x"], valid["y"])
            corr_label = "Spearman rho"
        else:
            r, p = pearsonr(valid["x"], valid["y"])
            corr_label = "r"
        p_txt = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        ax.text(
            0.97, 0.87,                     # ⬅ moved to right
            f"{corr_label} = {r:.2f}\n{p_txt}",
            transform=ax.transAxes,
            ha="right",                     # ⬅ align text to right
            va="top",
            fontsize=14,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                alpha=0.8,
                edgecolor="none"
            )
        )

def highlight_top_outliers(ax, df, xcol, ycol, label_col=None, top_n=3):
    valid = df.dropna(subset=[xcol, ycol])
    if valid.empty:
        return
    top = valid.nlargest(top_n, ycol)
    ax.scatter(
        top[xcol], top[ycol],
        s=80,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        zorder=4
    )
    if label_col and label_col in top.columns:
        for _, row in top.iterrows():
            ax.annotate(
                str(row[label_col])[:20],
                (row[xcol], row[ycol]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=10
            )


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# -----------------------------
# (a) Dangerous permissions vs PCLR
# -----------------------------
ax = axes[0]
if {"num_dangerous_permissions", "PCLR"}.issubset(fig_df.columns):
    plot_df = fig_df.dropna(subset=["num_dangerous_permissions", "PCLR"])
    colors = point_colors.loc[plot_df.index]

    ax.axhspan(0.5, 1, alpha=0.12)
    ax.axhspan(0.2, 0.5, alpha=0.06)

    ax.scatter(plot_df["num_dangerous_permissions"], plot_df["PCLR"],
               c=colors, alpha=0.65, s=30)

    add_regression_line(ax, plot_df["num_dangerous_permissions"], plot_df["PCLR"])
    add_corr_text(ax, plot_df["num_dangerous_permissions"], plot_df["PCLR"])
    highlight_top_outliers(ax, plot_df, "num_dangerous_permissions", "PCLR", "app_name")

    ax.set_xlabel("Dangerous permissions")
    ax.set_ylabel("PCLR")
    ax.set_title("(a) Permissions vs Pre-Consent Leakage")

# -----------------------------
# (b) ADII vs trackers
# -----------------------------
ax = axes[1]
if {"num_trackers", "ADII"}.issubset(fig_df.columns):
    plot_df = fig_df.dropna(subset=["num_trackers", "ADII"])
    colors = point_colors.loc[plot_df.index]

    q75 = plot_df["ADII"].quantile(0.75)
    ax.axhspan(q75, plot_df["ADII"].max(), alpha=0.08)

    ax.scatter(plot_df["num_trackers"], plot_df["ADII"],
               c=colors, alpha=0.5, s=28)

    add_regression_line(ax, plot_df["num_trackers"], plot_df["ADII"])
    add_corr_text(ax, plot_df["num_trackers"], plot_df["ADII"], method="spearman")
    highlight_top_outliers(ax, plot_df, "num_trackers", "ADII", "app_name")

    ax.set_xlabel("Number of trackers")
    ax.set_ylabel("ADII")
    ax.set_title("(b) Tracker Density vs Privacy Burden")

# -----------------------------
# (c) DGI vs permissions
# -----------------------------
ax = axes[2]
if {"num_permissions", "DGI"}.issubset(fig_df.columns):
    plot_df = fig_df.dropna(subset=["num_permissions", "DGI"])
    colors = point_colors.loc[plot_df.index]

    ax.axhspan(0.5, 1, alpha=0.1)

    ax.scatter(plot_df["num_permissions"], plot_df["DGI"],
               c=colors, alpha=0.5, s=28)

    add_regression_line(ax, plot_df["num_permissions"], plot_df["DGI"])
    add_corr_text(ax, plot_df["num_permissions"], plot_df["DGI"])
    highlight_top_outliers(ax, plot_df, "num_permissions", "DGI", "app_name")

    ax.set_xlabel("Number of permissions")
    ax.set_ylabel("DGI")
    ax.set_title("(c) Permission Load vs Disclosure Gap")

# -----------------------------
# Legend
# -----------------------------
if "region" in fig_df.columns:
    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=c, markersize=7)
        for c in region_color_map.values()
    ]
    labels = list(region_color_map.keys())
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=7,
        frameon=True,                     # ⬅ enable frame
        fontsize=14,
        handletextpad=0.3,
        columnspacing=0.8,
        handlelength=1.0,
        facecolor="lightgrey",            # ⬅ light grey fill
        edgecolor="none",                 # ⬅ no border (clean look)
        framealpha=0.3                    # ⬅ transparency (0–1)
    )

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(FIG_DIR / "combined_privacy_relationships.png", dpi=300, bbox_inches="tight")
plt.show()